# 트랙 노면 인식 · Drive 연동
같은 Google 계정의 내 드라이브/road_ai_v1을 사용합니다. 기본 실행은 저장된 clean 결과를 읽습니다. GPU는 새 학습을 할 때만 필요합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import tempfile, zipfile, json, shutil, subprocess, sys
drive.mount('/content/drive')
project = Path('/content/drive/MyDrive/road_ai_v1')
if not (project/'road_ai_starter.zip').is_file():
    raise FileNotFoundError(f'{project}에서 데이터 ZIP을 찾을 수 없습니다. Drive 계정을 확인하세요.')
work = Path(tempfile.mkdtemp(prefix='road_ai_', dir='/content'))
def unpack(archive, target, prefix=''):
    with zipfile.ZipFile(archive) as z:
        if any(Path(n).is_absolute() or '..' in Path(n).parts or not n.startswith(prefix) for n in z.namelist()):
            raise ValueError('잘못된 ZIP 경로')
        z.extractall(target)
print('Drive 연결됨:', project)

## 실행 선택
완료한 결과를 확인하려면 기본값 그대로 실행하세요. 다음 실험은 ACTION='train', STAGE='mild'로 바꾸고 GPU 런타임을 사용하세요. 학습마다 별도 폴더에 결과가 저장됩니다. FINAL_TEST는 단계를 선택한 뒤에만 사용하세요.

In [ ]:
ACTION = 'view'  # view / train
STAGE = 'clean'  # clean / mild / hard
EPOCHS = 12
FINAL_TEST = False
if ACTION not in ('view','train') or STAGE not in ('clean','mild','hard'):
    raise ValueError('ACTION 또는 STAGE를 확인하세요.')
archive_name = f'road_results_{STAGE}.zip'
if ACTION == 'view':
    candidates = sorted((project/'runs').glob(f'{STAGE}_*/{archive_name}'))
    archive = candidates[-1] if candidates else project/archive_name
    if not archive.is_file():
        raise FileNotFoundError(f'{STAGE} 결과가 없습니다. 새 학습은 ACTION=train을 선택하세요.')
    out = Path(tempfile.mkdtemp(dir=work))
    unpack(archive, out)
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('GPU 런타임에서 새 학습을 실행하세요.')
    unpack(project/'road_ai_starter.zip', work, 'road_ai_kit/')
    kit = work/'road_ai_kit'
    cmd = [sys.executable, '-u', 'train.py', '--stage', STAGE, '--epochs', str(EPOCHS)]
    if FINAL_TEST: cmd.append('--final-test')
    subprocess.run(cmd, cwd=kit, check=True)
    out = kit/'results'/STAGE
    local_zip = Path(shutil.make_archive(str(work/f'road_results_{STAGE}'), 'zip', out))
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    dest = project/'runs'/f'{STAGE}_{stamp}'
    dest.mkdir(parents=True, exist_ok=False)
    partial = dest/(archive_name+'.partial')
    shutil.copyfile(local_zip, partial)
    with zipfile.ZipFile(partial) as z:
        if z.testzip() is not None: raise IOError('Drive 결과 ZIP 확인 실패')
    shutil.copyfile(out/'metrics.json', dest/'metrics.json')
    archive = partial.rename(dest/archive_name)
print('Drive 결과:', archive)
report = json.loads((out/'metrics.json').read_text())
for probe, score in report['synthetic']['val'].items():
    print(probe, 'boundary F1', round(score['boundary_f1_2px'], 3), 'road IoU', round(score['road_iou'], 3))

## 실차 비교
왼쪽은 V135, 오른쪽은 CNN입니다. 다음 분석에는 위에 표시된 Drive 결과와 metrics.json을 사용합니다.

In [ ]:
from IPython.display import display
from PIL import Image
for path in sorted(out.glob('real_*.jpg')):
    print(path.name)
    display(Image.open(path))